# 05 · RoBERTa fine-tuning for Empathy prediction (Days 11-13)

Fine-tune RoBERTa-base on the WASSA CONV-Turn Empathy target and compare against
the Ridge + TF-IDF baseline established in `04_tfidf_sweeps.ipynb`.

**Approach.** RoBERTa (Liu et al., 2019) is a transformer-based language model
pretrained on ~160GB of English text using a masked language modeling objective.
Unlike the bag-of-words TF-IDF representation used in the baseline, RoBERTa
produces contextualized token embeddings that capture word meaning in context.
We adapt the pretrained model to Empathy prediction by attaching a regression
head to the `[CLS]` token output and fine-tuning all parameters on the training
split.

**Setup.** Fine-tuning is orchestrated via Hugging Face's `Trainer` API, which
handles the training loop, evaluation, checkpointing, and metric computation.
Hyperparameters follow standard practice for BERT-family fine-tuning:
learning rate 2e-5, batch size 16, 3 epochs, AdamW optimizer with linear warmup.
The checkpoint with the highest development-set Pearson correlation is retained
as the final model.

**Split note.** This notebook uses the same internal conversation-grouped
70/15/15 split as the baseline notebooks, kept for consistency across the
Ridge–RoBERTa comparison. All model-selection decisions are made on dev; test
is used only for final reporting.

In [ ]:
!git clone https://github.com/DavorSopar/dataset-analysis.git /content/dataset-analysis
import sys
sys.path.insert(0, '/content/dataset-analysis/src')

Cloning into '/content/dataset-analysis'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 35 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.41 MiB | 6.91 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

In [ ]:
# Load and resolve the two missing SelfDisclosure values (speaker-mean; see 01_eda).
df = impute_selfdisclosure(load_convt())
print(f"{len(df):,} turns across {df['conversation_id'].nunique()} conversations")
print("Targets:", TARGETS)

11,166 turns across 487 conversations
Targets: ['Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


In [ ]:
groups = df["conversation_id"].to_numpy()

# 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(df, groups=groups))

# split temp 50/50 -> 15% dev, 15% test (still grouped)
temp = df.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
dev_rel, test_rel = next(gss2.split(temp, groups=temp["conversation_id"].to_numpy()))
dev_idx, test_idx = temp_idx[dev_rel], temp_idx[test_rel]

train, dev, test = df.iloc[train_idx], df.iloc[dev_idx], df.iloc[test_idx]

# Guarantee no conversation appears in more than one split.
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    "turns": [len(train), len(dev), len(test)],
    "conversations": [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=["train", "dev", "test"])
split_tbl["turns_%"] = (100 * split_tbl["turns"] / len(df)).round(1)
print(split_tbl)
print("\nNo conversation overlaps across splits. Good.")

       turns  conversations  turns_%
train   7788            340  69.7000
dev     1640             73  14.7000
test    1738             74  15.6000

No conversation overlaps across splits. Good.


In [ ]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

## 1 · Preprocessing — tokenization and dataset formatting

RoBERTa expects input as sequences of subword token IDs from its fixed
pretrained vocabulary (~50,000 subwords). Text preprocessing consists of two
steps: (i) tokenize each turn into `input_ids` and `attention_mask` using
RoBERTa's matched pretrained tokenizer, and (ii) convert the pandas DataFrames
into Hugging Face `Dataset` objects with the target column renamed to `labels`
(the field name expected by `Trainer`). Sequences are truncated at 128 tokens,
which covers essentially all conversation turns in the WASSA data without
padding overhead.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

# Convert pandas → HF Dataset, rename Empathy to labels (Trainer expects "labels")
train_ds = Dataset.from_pandas(train[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))
dev_ds   = Dataset.from_pandas(dev[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))
test_ds  = Dataset.from_pandas(test[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))

# Tokenize all splits
train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds   = dev_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

# Ensure labels are float32 (regression needs floats, not ints)
import torch
train_ds = train_ds.map(lambda x: {"labels": float(x["labels"])})
dev_ds   = dev_ds.map(lambda x: {"labels": float(x["labels"])})
test_ds  = test_ds.map(lambda x: {"labels": float(x["labels"])})

# Sanity check
print(train_ds[0])

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1738 [00:00<?, ? examples/s]

Map:   0%|          | 0/7788 [00:00<?, ? examples/s]

Map:   0%|          | 0/1640 [00:00<?, ? examples/s]

Map:   0%|          | 0/1738 [00:00<?, ? examples/s]

{'text': 'I feel very sad for the people.', 'labels': 3.3333, '__index_level_0__': 22, 'input_ids': [0, 100, 619, 182, 5074, 13, 5, 82, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(f"train: {len(train_ds)}, dev: {len(dev_ds)}, test: {len(test_ds)}")

train: 7788, dev: 1640, test: 1738


## 2 · Model and Trainer configuration

The model is loaded via `AutoModelForSequenceClassification.from_pretrained`
with `num_labels=1` and `problem_type="regression"`, which attaches a
randomly-initialized regression head on top of the pretrained transformer body.
During fine-tuning, both the pretrained body and the new head are updated: the
head learns from scratch (starting from random weights), while the body is
gently adjusted from its pretrained state by using a small learning rate (2e-5).
This preserves the general language understanding acquired during pretraining
while adapting the model's outputs to the empathy prediction task.

`DataCollatorWithPadding` handles per-batch dynamic padding, avoiding the waste
of padding all sequences to a fixed dataset-wide maximum. A custom
`compute_metrics` function reports MAE, RMSE, and Pearson r after each
evaluation pass; the best checkpoint is selected by Pearson correlation on the
development set.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
import numpy as np

# Load the model with a regression head
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=1,
    problem_type="regression",
)

# Data collator handles per-batch padding automatically
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metrics function — same three you've been using
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()  # shape (n, 1) → (n,)
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    r = pearsonr(labels, predictions)[0]
    return {"mae": mae, "rmse": rmse, "pearson": r}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta_empathy",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="pearson",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",  # disables wandb/tensorboard reporting
)

# Assemble the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Sanity check — this should print without errors
print("Trainer is ready. Do not call trainer.train() yet.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer is ready. Do not call trainer.train() yet.


## 3 · Training

Fine-tuning runs for three epochs on the training split, with evaluation on the
development split after each epoch. Both training loss and development metrics
are logged to monitor for overfitting.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse,Pearson
1,0.476857,0.500314,0.554565,0.707329,0.693686
2,0.397831,0.429922,0.509101,0.655684,0.710795
3,0.300926,0.422114,0.504834,0.649703,0.723950


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1461, training_loss=0.4406161311552347, metrics={'train_runtime': 408.4409, 'train_samples_per_second': 57.203, 'train_steps_per_second': 3.577, 'total_flos': 752256530777424.0, 'train_loss': 0.4406161311552347, 'epoch': 3.0})

## 4 · Final evaluation

Evaluate the best-checkpoint model on both dev (as a reference) and test (the
number reported in the thesis Results section). Test is held out and touched
only once, at this stage.

In [ ]:
# Evaluate on dev
dev_results = trainer.evaluate(dev_ds)
print("DEV:", dev_results)

# Evaluate on test
test_results = trainer.evaluate(test_ds)
print("TEST:", test_results)

Training Loss,Validation Loss,Epoch,Mae,Rmse,Pearson
0.300926,0.422114,3,0.504834,0.649703,0.723950


DEV: {'eval_loss': 0.4221137464046478, 'eval_mae': 0.5048344135284424, 'eval_rmse': 0.6497028368469466, 'eval_pearson': 0.723950207233429}


Training Loss,Validation Loss,Epoch,Mae,Rmse,Pearson
0.300926,0.509952,3,0.560896,0.714109,0.697389


TEST: {'eval_loss': 0.5099517703056335, 'eval_mae': 0.5608959197998047, 'eval_rmse': 0.7141090745156747, 'eval_pearson': 0.6973892450332642}
